# K-Means聚类算法实现

K-Means是最经典的无监督学习算法，通过迭代优化将数据分配到K个簇中。

## 1. 导入必要的库

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

## 2. 实现K-Means类

In [ ]:
class KMeans:
    """K-Means聚类算法"""
    
    def __init__(self, n_clusters=3, max_iters=100, init_method='random', random_state=None):
        self.n_clusters = n_clusters
        self.max_iters = max_iters
        self.init_method = init_method
        self.random_state = random_state
        self.centroids = None
        self.labels = None
        self.inertia_ = None
    
    def _init_centroids(self, X):
        """初始化聚类中心"""
        n_samples = X.shape[0]
        
        if self.random_state is not None:
            np.random.seed(self.random_state)
        
        if self.init_method == 'random':
            indices = np.random.choice(n_samples, self.n_clusters, replace=False)
            return X[indices]
        
        elif self.init_method == 'kmeans++':
            centroids = [X[np.random.randint(n_samples)]]
            
            for _ in range(1, self.n_clusters):
                distances = np.array([
                    min([np.linalg.norm(x - c)**2 for c in centroids])
                    for x in X
                ])
                
                probabilities = distances / distances.sum()
                cumulative_probs = probabilities.cumsum()
                r = np.random.rand()
                
                for j, p in enumerate(cumulative_probs):
                    if r < p:
                        centroids.append(X[j])
                        break
            
            return np.array(centroids)
    
    def _assign_clusters(self, X, centroids):
        """E步：分配簇"""
        distances = np.zeros((X.shape[0], self.n_clusters))
        
        for i in range(self.n_clusters):
            distances[:, i] = np.linalg.norm(X - centroids[i], axis=1)
        
        return np.argmin(distances, axis=1)
    
    def _update_centroids(self, X, labels):
        """M步：更新聚类中心"""
        centroids = np.zeros((self.n_clusters, X.shape[1]))
        
        for i in range(self.n_clusters):
            cluster_points = X[labels == i]
            if len(cluster_points) > 0:
                centroids[i] = cluster_points.mean(axis=0)
            else:
                centroids[i] = X[np.random.randint(X.shape[0])]
        
        return centroids
    
    def _compute_inertia(self, X, labels, centroids):
        """计算簇内误差平方和"""
        inertia = 0.0
        for i in range(self.n_clusters):
            cluster_points = X[labels == i]
            if len(cluster_points) > 0:
                inertia += np.sum((cluster_points - centroids[i])**2)
        return inertia
    
    def fit(self, X, verbose=False):
        """训练K-Means模型"""
        self.centroids = self._init_centroids(X)
        self.centroids_history = [self.centroids.copy()]
        self.inertia_history = []
        
        for iteration in range(self.max_iters):
            # E步：分配簇
            labels = self._assign_clusters(X, self.centroids)
            
            # M步：更新中心
            new_centroids = self._update_centroids(X, labels)
            
            # 计算Inertia
            inertia = self._compute_inertia(X, labels, new_centroids)
            self.inertia_history.append(inertia)
            
            if verbose and (iteration + 1) % 10 == 0:
                print(f"Iteration {iteration+1}/{self.max_iters}, Inertia: {inertia:.4f}")
            
            # 检查收敛
            if np.allclose(self.centroids, new_centroids, rtol=1e-6):
                if verbose:
                    print(f"Converged at iteration {iteration+1}")
                self.centroids = new_centroids
                self.labels = labels
                break
            
            self.centroids = new_centroids
            self.centroids_history.append(self.centroids.copy())
        
        self.labels = self._assign_clusters(X, self.centroids)
        self.inertia_ = self._compute_inertia(X, self.labels, self.centroids)
        
        return self
    
    def predict(self, X):
        """预测新数据的簇标签"""
        return self._assign_clusters(X, self.centroids)

## 3. 生成测试数据

生成3个高斯分布的簇。

In [ ]:
# 生成3个簇的数据
n_samples = 300
centers = np.array([[1, 1], [5, 5], [9, 1]])

X1 = np.random.randn(n_samples // 3, 2) * 0.8 + centers[0]
X2 = np.random.randn(n_samples // 3, 2) * 0.8 + centers[1]
X3 = np.random.randn(n_samples // 3, 2) * 0.8 + centers[2]

X = np.vstack([X1, X2, X3])

# 可视化原始数据
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], alpha=0.6, edgecolors='k', s=50)
plt.xlabel('特征 1')
plt.ylabel('特征 2')
plt.title('原始数据分布')
plt.grid(True, alpha=0.3)
plt.show()

print(f"数据形状: {X.shape}")
print(f"数据范围: X1=[{X[:, 0].min():.2f}, {X[:, 0].max():.2f}], "
      f"X2=[{X[:, 1].min():.2f}, {X[:, 1].max():.2f}]")

## 4. 训练K-Means模型

In [ ]:
# 创建并训练K-Means模型
kmeans = KMeans(n_clusters=3, max_iters=100, init_method='kmeans++', random_state=42)
kmeans.fit(X, verbose=True)

print(f"\n最终Inertia: {kmeans.inertia_:.4f}")
print(f"\n聚类中心:\n{kmeans.centroids}")

## 5. 可视化聚类结果

In [ ]:
# 绘制聚类结果
plt.figure(figsize=(10, 7))

# 绘制数据点
scatter = plt.scatter(X[:, 0], X[:, 1], c=kmeans.labels, cmap='viridis',
                     alpha=0.6, edgecolors='k', s=50)

# 绘制聚类中心
plt.scatter(kmeans.centroids[:, 0], kmeans.centroids[:, 1],
           c='red', marker='X', s=300, edgecolors='k', linewidths=2,
           label='聚类中心')

plt.xlabel('特征 1')
plt.ylabel('特征 2')
plt.title('K-Means聚类结果 (K=3)')
plt.colorbar(scatter, label='簇标签')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. 可视化K-Means迭代过程

In [ ]:
# 显示关键迭代步骤
n_frames = min(6, len(kmeans.centroids_history))
frames = np.linspace(0, len(kmeans.centroids_history) - 1, n_frames, dtype=int)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, frame in enumerate(frames):
    ax = axes[idx]
    centroids = kmeans.centroids_history[frame]
    labels = kmeans._assign_clusters(X, centroids)
    
    # 绘制数据点
    scatter = ax.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis',
                        alpha=0.6, edgecolors='k', s=50)
    
    # 绘制聚类中心
    ax.scatter(centroids[:, 0], centroids[:, 1],
              c='red', marker='X', s=200, edgecolors='k', linewidths=2)
    
    ax.set_xlabel('特征 1')
    ax.set_ylabel('特征 2')
    ax.set_title(f'迭代 {frame + 1}')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Inertia收敛曲线

In [ ]:
# 绘制Inertia随迭代次数的变化
plt.figure(figsize=(10, 6))
plt.plot(kmeans.inertia_history, 'b-o', linewidth=2, markersize=6)
plt.xlabel('迭代次数')
plt.ylabel('Inertia (簇内误差平方和)')
plt.title('K-Means收敛过程')
plt.grid(True, alpha=0.3)
plt.show()

print(f"初始Inertia: {kmeans.inertia_history[0]:.4f}")
print(f"最终Inertia: {kmeans.inertia_history[-1]:.4f}")
print(f"改善: {(kmeans.inertia_history[0] - kmeans.inertia_history[-1]):.4f}")

## 8. 肘部法则选择K值

通过绘制K与Inertia的关系，找到曲线的"肘部"来选择最佳K值。

In [ ]:
# 测试不同的K值
max_k = 10
inertias = []

for k in range(1, max_k + 1):
    kmeans_test = KMeans(n_clusters=k, random_state=42)
    kmeans_test.fit(X)
    inertias.append(kmeans_test.inertia_)
    print(f"K={k}: Inertia={kmeans_test.inertia_:.4f}")

# 绘制肘部曲线
plt.figure(figsize=(10, 6))
plt.plot(range(1, max_k + 1), inertias, 'bo-', linewidth=2, markersize=8)
plt.xlabel('聚类数量 K')
plt.ylabel('Inertia (簇内误差平方和)')
plt.title('肘部法则选择K值')
plt.grid(True, alpha=0.3)
plt.xticks(range(1, max_k + 1))
plt.axvline(x=3, color='r', linestyle='--', alpha=0.5, label='K=3')
plt.legend()
plt.show()

## 9. 对比不同初始化方法

比较随机初始化和K-Means++初始化的效果。

In [ ]:
methods = ['random', 'kmeans++']
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, method in enumerate(methods):
    ax = axes[idx]
    
    # 训练模型
    kmeans_test = KMeans(n_clusters=3, init_method=method, random_state=42)
    kmeans_test.fit(X)
    
    # 绘制结果
    scatter = ax.scatter(X[:, 0], X[:, 1], c=kmeans_test.labels,
                        cmap='viridis', alpha=0.6, edgecolors='k', s=50)
    ax.scatter(kmeans_test.centroids[:, 0], kmeans_test.centroids[:, 1],
              c='red', marker='X', s=300, edgecolors='k', linewidths=2)
    
    ax.set_xlabel('特征 1')
    ax.set_ylabel('特征 2')
    ax.set_title(f'{method} 初始化\nInertia={kmeans_test.inertia_:.2f}')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. 测试预测功能

In [ ]:
# 生成新的测试点
X_new = np.array([[2, 2], [6, 6], [8, 1], [0, 0]])
predictions = kmeans.predict(X_new)

print("预测新数据点的簇标签：")
for i, (point, label) in enumerate(zip(X_new, predictions)):
    print(f"  点 {i+1}: {point} -> 簇 {label}")

# 可视化
plt.figure(figsize=(10, 7))

# 原始数据
plt.scatter(X[:, 0], X[:, 1], c=kmeans.labels, cmap='viridis',
           alpha=0.4, edgecolors='k', s=50, label='训练数据')

# 聚类中心
plt.scatter(kmeans.centroids[:, 0], kmeans.centroids[:, 1],
           c='red', marker='X', s=300, edgecolors='k', linewidths=2,
           label='聚类中心')

# 新数据点
plt.scatter(X_new[:, 0], X_new[:, 1], c=predictions, cmap='viridis',
           marker='s', s=200, edgecolors='red', linewidths=3,
           label='新数据点')

plt.xlabel('特征 1')
plt.ylabel('特征 2')
plt.title('预测新数据点的簇标签')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 11. 不同K值的聚类效果对比

In [ ]:
# 测试K=2,3,4,5的聚类效果
k_values = [2, 3, 4, 5]
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for idx, k in enumerate(k_values):
    ax = axes[idx]
    
    # 训练模型
    kmeans_k = KMeans(n_clusters=k, init_method='kmeans++', random_state=42)
    kmeans_k.fit(X)
    
    # 绘制结果
    scatter = ax.scatter(X[:, 0], X[:, 1], c=kmeans_k.labels,
                        cmap='viridis', alpha=0.6, edgecolors='k', s=50)
    ax.scatter(kmeans_k.centroids[:, 0], kmeans_k.centroids[:, 1],
              c='red', marker='X', s=300, edgecolors='k', linewidths=2)
    
    ax.set_xlabel('特征 1')
    ax.set_ylabel('特征 2')
    ax.set_title(f'K={k}, Inertia={kmeans_k.inertia_:.2f}')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12. 应用示例：图像压缩

使用K-Means进行颜色量化，实现简单的图像压缩。

In [ ]:
# 创建一个简单的彩色图像
image = np.random.rand(50, 50, 3)

# 将图像reshape为(n_pixels, 3)
pixels = image.reshape(-1, 3)

# 使用K-Means压缩颜色到16种
n_colors = 16
kmeans_img = KMeans(n_clusters=n_colors, init_method='kmeans++', random_state=42)
kmeans_img.fit(pixels)

# 用聚类中心替换原始颜色
compressed_pixels = kmeans_img.centroids[kmeans_img.labels]
compressed_image = compressed_pixels.reshape(image.shape)

# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.imshow(image)
ax1.set_title(f'原始图像 (全彩色)')
ax1.axis('off')

ax2.imshow(compressed_image)
ax2.set_title(f'压缩图像 ({n_colors}种颜色)')
ax2.axis('off')

plt.tight_layout()
plt.show()

print(f"压缩率: {(1 - n_colors / (50*50)) * 100:.2f}%")

## 13. 总结

通过本notebook，我们实现并演示了K-Means聚类算法：

1. **基础实现**：从零实现K-Means算法
2. **初始化方法**：对比random和K-Means++初始化
3. **可视化**：展示迭代过程和收敛曲线
4. **肘部法则**：使用Inertia曲线选择最佳K值
5. **应用示例**：图像压缩

**关键要点**：
- K-Means是EM算法的应用：E步分配，M步更新
- K-Means++初始化能显著改善结果
- 肘部法则是选择K值的实用方法
- 算法简单高效，但对初始值和异常值敏感
- 适用于凸形、大小相似的簇